In [ ]:
import os

import ee

from utils import (
    get_export_periods,
    get_export_tile_ids,
    get_wgs84_region_bounds,
    get_region_pixel_size,
    get_region_crs,
    get_region_transform,
    gee_grid_tiles,
    get_long_region_name,
    get_long_ds_name,
    get_nodata_value
)

First, we must retrieve cloud bucket access credentials from the corresponding JSON file.

In [ ]:
# Replace with your equivalent path to credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    "path/to/your/credentials.json"
)

In [ ]:
ee.Authenticate()

In [ ]:
# Replace with your equivalent cloud project
ee.Initialize(project="your-cloud-project")

# Input arguments

In [ ]:
# Bucket name
bucket = "bucket_name"

# Region
short_region_name = "est"

# Dataset
short_ds_name = "s2"

Uncomment all years in order to process the whole time series.

In [ ]:
# List of years to process
years = [
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]

In [ ]:
# Dictionary of time periods to export
export_periods = get_export_periods(years, short_region_name)

In [ ]:
print(export_periods)

You can uncomment the next cell if you wish to test the workflow with a single season instead of the full list.

In [ ]:
export_periods = {}
for year in years:
    export_periods[year] = {
        "start_dates": [
            # f"{year}-04-01",
            f"{year}-06-01",
            # f"{year}-09-01"
        ],
        "end_dates": [
            # f"{year}-05-31",
            f"{year}-08-31",
            # f"{year}-10-31"
        ]
    }

Uncomment the variables you would like to process.

In [ ]:
# List of variables to calculate
variables = [
    "ndvi",
    "evi",
    "savi",
    "fvc",
    "ndwi",
    "bsi",
    "ndmi",
    # "wiw",
    "lai",
    "gndvi",
    # "lai_savi",
    # "spinr_ndwi1",
    # "b1",
    # "b2",
    # "b3",
    # "b4",
    # "b5",
    # "b6",
    # "b7",
    # "b8",
    # "b8a",
    # "b9",
    # "b11",
    # "b12"
]

In [ ]:
# List of statistics to calculate
stats = [
    # "median",
    # "mean",
    "std",
    # "min",
    # "max"
]

In [ ]:
# Dictionary mapping statistic keys to reducer functions
stat_reducers = {
    "median": lambda col: col.median(),
    "mean": lambda col: col.mean(),
    "std": lambda col: col.reduce(ee.Reducer.stdDev()),
    "min": lambda col: col.min(),
    "max": lambda col: col.max()
}

Tile IDs that are going to be exported are given in the following list. Only the tiles that intersect with the boundaries of the corresponding region (Estonia, Baltic states or European countries) will be exported.

In [ ]:
# Tile IDs to export
export_tile_ids = get_export_tile_ids(short_region_name)

In [ ]:
print(export_tile_ids)

You can uncomment the next cell if you wish to test the workflow with a single tile or a couple of tiles instead of the full list.

In [ ]:
# export_tile_ids = ["09"]

# Create output grid tiles

We will create an output grid where each tile is 10000$\times$10000 pixels, which is the default maximum output image size in GEE. This means that for 10 m resolution the `scale` parameter will be 100000 (100 km), for 30 m resolution 300000 (300 km) and for 100 m resolution 1000000 (1000 km).

In [ ]:
# Region bounds in WGS84
region_bounds = get_wgs84_region_bounds(short_region_name)

# Pixel size
pixel_size = get_region_pixel_size(short_region_name)

# Region CRS
region_crs = get_region_crs(short_region_name)

# Calculate transform
region_transform = get_region_transform(short_region_name)

# Generate grid tiles
grid_tiles = gee_grid_tiles(region_bounds, region_crs, pixel_size)

In [ ]:
print(region_crs)
print(region_transform)

# Calculate indices and export tiles

In [ ]:
# Mask pixels based on probability of being clear
def mask_low_qa(image, qa_band, clear_threshold=0.5):
  mask = image.select(qa_band).gte(clear_threshold)
  return image.updateMask(mask)

In [ ]:
# Apply preprocessing steps to Sentinel-2 collection
def preprocess_s2_collection(start_date, end_date, grid_tiles, variable):

    # Define the required bands for each index
    required_bands = {
        "ndvi": ["B4", "B8"],
        "evi": ["B2", "B4", "B8"],
        "savi": ["B8", "B4"],
        "fvc": ["B4", "B8"],
        "ndwi": ["B3", "B8"],
        "bsi": ["B11", "B4", "B8", "B2"],
        "ndmi": ["B8", "B11"],
        "wiw": ["B8A", "B12"],
        "lai": ["B2", "B4", "B8"],
        "gndvi": ["B8", "B3"],
        "lai_savi": ["B8", "B4"],
        "spinr_ndwi1": ["B8A", "B11"],
        "b1": ["B1"],
        "b2": ["B2"],
        "b3": ["B3"],
        "b4": ["B4"],
        "b5": ["B5"],
        "b6": ["B6"],
        "b7": ["B7"],
        "b8": ["B8"],
        "b8a": ["B8A"],
        "b9": ["B9"],
        "b11": ["B11"],
        "b12": ["B12"]
    }
    
    # Read Sentinel-2 collection
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    
    # Filter Sentinel-2 based on time and bounds
    s2_filtered = s2\
        .filterDate(
            ee.Date(start_date), ee.Date(end_date).advance(1, "day")
        )\
        .filterBounds(grid_tiles.geometry())
    
    # Set the desired bands based on the requested index
    desired_bands = required_bands.get(variable.lower(), [])
    s2_filtered_reordered = s2_filtered\
        .map(lambda image: image.select(desired_bands))
    
    # Read Cloud Score+ collection
    cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    cs_plus_bands = ["cs", "cs_cdf"]
    
    # Add Cloud Score+ bands to Sentinel-2
    s2_with_cs_plus = s2_filtered_reordered\
        .linkCollection(cs_plus, cs_plus_bands)
    
    # Mask clouds based on Cloud Score+
    qa_band = "cs"
    clear_threshold = 0.5
    s2_masked = s2_with_cs_plus\
        .map(lambda img: mask_low_qa(img, qa_band, clear_threshold))
    
    return s2_masked

In [ ]:
# Calculate spectral index based on variable name for Sentinel-2 collection
def calc_s2_index_band(s2_collection, variable):
    
    # Normalize input to lowercase for consistency
    variable = variable.lower()

    # Skip index calculation for raw bands
    if variable in [
        "b1", "b2", "b3", "b4", "b5", "b6", 
        "b7", "b8", "b8a", "b9", "b11", "b12"
    ]:
        index_band = (
            s2_collection
            .select(variable.upper())
            .rename(variable)
        )
    
    # Normalized Difference Vegetation Index (NDVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndvi/
    elif variable == "ndvi":
        
        # Formula: (NIR - RED) / (NIR + RED)
        index_band = s2_collection\
            .normalizedDifference(["B8", "B4"])\
            .rename(variable)
        
    # Enhanced Vegetation Index (EVI)
    elif variable.lower() == "evi":
        
        # Formula: 2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))
        evi = s2_collection\
            .expression(
                "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
                {
                    "NIR": s2_collection.select("B8").divide(10000),
                    "RED": s2_collection.select("B4").divide(10000),
                    "BLUE": s2_collection.select("B2").divide(10000)
                }
            )

        # Constrain EVI to a realistic range
        lower_limit = -1
        upper_limit = 1
        index_band = (
            evi
            .where(evi.lt(lower_limit), lower_limit)
            .where(evi.gt(upper_limit), upper_limit)
            .rename(variable)
        )

    # Soil-Adjusted Vegetation Index (SAVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/savi/
    elif variable == "savi":
        
        """
        Formula: ((NIR - RED) / (NIR + RED + L)) * (1 + L)
        
        Soil brightness correction factor (L) is defined as 0.5 to 
        accommodate most land cover types
        """
        savi = s2_collection.expression(
            f"((NIR - RED) / (NIR + RED + 0.5)) * (1 + 0.5)",
            {
                "NIR": s2_collection.select("B8").divide(10000),
                "RED": s2_collection.select("B4").divide(10000)
            }
        )
        index_band = savi.rename(variable)

    # Fractional Vegetation Cover (FVC)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/fcover/
    elif variable == "fvc":
        
        # Compute NDVI first
        ndvi = calc_s2_index_band(s2_collection, "ndvi")
        
        """
        Formula: (NDVI - NDVIs) / (NDVIv - NDVIs)
        
        Default values for NDVIs and NDVIv are taken from 
        https://custom-scripts.sentinel-hub.com/custom-scripts/landsat-8/land_surface_temperature_mapping/
        """
        ndvis = 0.2
        ndviv = 0.8
        fvc = ndvi.expression(
            "(NDVI - NDVIs) / (NDVIv - NDVIs)",
            {
                "NDVI": ndvi,
                "NDVIs": ndvis,
                "NDVIv": ndviv
            }
        )
        
        # Constrain FVC to a realistic range
        lower_limit = 0
        upper_limit = 1
        index_band = (
            fvc
            .where(fvc.lt(lower_limit), lower_limit)
            .where(fvc.gt(upper_limit), upper_limit)
            .rename(variable)
        )
        
    # Normalized Difference Water Index (NDWI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndwi/
    elif variable == "ndwi":
        
        # Formula: (GREEN - NIR) / (GREEN + NIR)
        index_band = s2_collection\
            .normalizedDifference(["B3", "B8"])\
            .rename(variable)

    # Bare Soil Index (BSI)
    # https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/barren_soil/
    elif variable == "bsi":
        
        # Formula: ((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))
        index_band = s2_collection.expression(
            "((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))",
            {
                "B11": s2_collection.select("B11").divide(10000),
                "B4": s2_collection.select("B4").divide(10000),
                "B8": s2_collection.select("B8").divide(10000),
                "B2": s2_collection.select("B2").divide(10000)
            }
        ).rename(variable)
        
    # Normalized Difference Moisture Index (NDMI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/ndmi/
    elif variable == "ndmi":
        
        # Formula: (NIR - SWIR) / (NIR + SWIR)
        index_band = s2_collection\
            .normalizedDifference(["B8", "B11"])\
            .rename(variable)

    # Wetlands Index for Water (WIW)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/wiw_s2_script/
    elif variable == "wiw":
        
        # Formula: 1 if (B8A <= 0.1804) and (B12 <= 0.1131), else 0
        nodata_mask = s2_collection.select(["B8A", "B12"]).mask()\
            .reduce(ee.Reducer.min())
        index_band = s2_collection.expression(
            "(B8A <= 0.1804) and (B12 <= 0.1131) ? 1 : 0",
            {
                "B8A": s2_collection.select("B8A").divide(10000),
                "B12": s2_collection.select("B12").divide(10000)
            }
        ).rename(variable)
        
        # Apply the nodata mask and fill unmasked areas with nodata value
        nodata_val = 255
        index_band = index_band.updateMask(nodata_mask)\
            .unmask(value=nodata_val)

    # Leaf Area Index (LAI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/lai/
    elif variable == "lai":
        
        """
        Formula: LAI = (3.618 * EVI) - 0.118
        Based on the formula from Boegh et al. (2002), DOI: 
        https://doi.org/10.1016/S0034-4257(01)00342-X.
        Value range based on Boegh et al. (2002), DOI: 
        https://doi.org/10.1046/j.1466-822X.2003.00026.x.
        """
        evi = s2_collection\
            .expression(
                "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
                {
                    "NIR": s2_collection.select("B8").divide(10000),
                    "RED": s2_collection.select("B4").divide(10000),
                    "BLUE": s2_collection.select("B2").divide(10000)
                }
            )
        lai = evi.expression(
            "(3.618 * EVI) - 0.118",
            {
                "EVI": evi
            }
        )
        
        # Constrain LAI to a realistic range
        lower_limit = 0
        upper_limit = 47
        index_band = (
            lai
            .where(lai.lt(lower_limit), lower_limit)
            .where(lai.gt(upper_limit), upper_limit)
            .rename(variable)
        )
        
    # Green Normalized Difference Vegetation Index (GNDVI)
    # https://custom-scripts.sentinel-hub.com/sentinel-2/gndvi/
    elif variable == "gndvi":
        
        # Formula: (NIR - GREEN) / (NIR + GREEN)
        index_band = s2_collection\
            .normalizedDifference(["B8", "B3"])\
            .rename(variable)

    # LAI_SAVI
    # https://github.com/jbferet/spinR/blob/master/R/Lib_SpectralIndices.R
    elif variable == "lai_savi":
        
        """
        Formula (spinR):
        LAI_SAVI = -log(0.371 + 1.5 * SAVI) / 2.4
        
        SAVI = ((NIR - RED) / (NIR + RED + L)) * (1 + L)
        where L = 0.5
        
        Reuses the existing SAVI calculation branch.
        """
        savi = calc_s2_index_band(s2_collection, "savi")
        lai_savi = savi.expression(
            "-log(0.371 + 1.5 * SAVI) / 2.4",
            {
                "SAVI": savi
            }
        )
        index_band = lai_savi.rename(variable)

    # spinR NDWI1
    # https://github.com/jbferet/spinR/blob/master/R/Lib_SpectralIndices.R
    elif variable == "spinr_ndwi1":
        
        # Formula: (B8A - B11) / (B8A + B11)
        index_band = s2_collection\
            .normalizedDifference(["B8A", "B11"])\
            .rename(variable)
        
    return index_band

In [ ]:
# List of grid IDs
grid_ids = grid_tiles.aggregate_array("system:index").getInfo()

In [ ]:
# Load land mask from asset
if short_region_name == "est":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "est_ref_grid_500m_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "bal":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "bal_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)
elif short_region_name == "eur":
    land_mask_path = (
        "projects/ee-holgervirro/assets/"
        "eur_ref_grid_1km_buffer_2025"
    )
    land_mask = ee.Image(land_mask_path)

In [ ]:
# Loop over years
for year in export_periods.keys():
    
    print(f"Year being processed: {year}")

    # Start dates
    start_dates = export_periods[year]["start_dates"]

    # End dates
    end_dates = export_periods[year]["end_dates"]

    # Loop over variables
    for variable in variables:
        
        print(f"\tVariable being processed: {variable}")

        # Nodata value
        nodata_val = get_nodata_value(variable, short_region_name)

        # Preprocess annual collection
        s2_preprocessed_annual = preprocess_s2_collection(
            f"{year}-01-01", f"{year}-12-31", grid_tiles, variable
        )

        # Resample annual collection based on region
        if short_region_name in ["bal", "eur"]:
            if variable != "wiw":
                s2_preprocessed_annual = (
                    s2_preprocessed_annual
                    .map(lambda img: img.resample("bilinear"))
                )

        # Calculate index per image in annual collection
        index_collection_annual = (
            s2_preprocessed_annual
            .map(lambda img: calc_s2_index_band(img, variable))
            .select(variable)
        )

        # Loop over statistics
        for stat in stats:

            print(f"\t\tStatistic being processed: {stat}")
            
            # Get reducer function
            reducer_fn = stat_reducers[stat]

            # Calculate annual statistic
            index_annual_stat = (
                reducer_fn(index_collection_annual)
                .rename(f"{variable}_{stat}")
                .unmask(value=nodata_val)
            )

            # Loop over time periods
            for start_date, end_date in zip(start_dates, end_dates):
                
                print(f"\t\t\tTime period start date: {start_date}")
                print(f"\t\t\tTime period end date: {end_date}")

                # Preprocess seasonal collection
                s2_preprocessed = preprocess_s2_collection(
                    start_date, end_date, grid_tiles, variable
                )

                # Resample seasonal collection based on region
                if short_region_name in ["bal", "eur"]:
                    if variable != "wiw":
                        s2_preprocessed = (
                            s2_preprocessed
                            .map(lambda img: img.resample("bilinear"))
                        )

                # Calculate index per image in seasonal collection
                index_collection_seasonal = (
                    s2_preprocessed
                    .map(lambda img: calc_s2_index_band(img, variable))
                    .select(variable)
                )

                # Calculate seasonal statistic
                index_season_stat = (
                    reducer_fn(index_collection_seasonal)
                    .rename(f"{variable}_{stat}")
                )

                # Create a mask for nodata pixels
                nodata_mask = (
                    index_season_stat
                    .unmask(value=nodata_val)
                    .eq(nodata_val)
                )
                
                # Fill masked pixels with annual values
                image_filled = (
                    index_season_stat
                    .unmask(index_annual_stat)
                    .where(
                        index_season_stat.eq(nodata_val),
                        index_annual_stat
                    )
                )

                # Replace masked pixels with the nodata value
                unmasked_image = image_filled.unmask(value=nodata_val)

                # Skip scaling for raw bands
                if variable in [
                    "b1", "b2", "b3", "b4", "b5", "b6",
                    "b7", "b8", "b8a", "b9", "b11", "b12"
                ]:
                    out_image = unmasked_image.toInt16()
                
                # Skip scaling for binary WIW image
                elif variable == "wiw":
                    out_image = unmasked_image
                
                else:
                    
                    # Scale values
                    out_image = (
                        unmasked_image.where(nodata_mask, 0)
                        .multiply(1000)
                        .round()
                    )
                    if variable == "lai":
                        out_image = out_image.toInt32()
                    else:
                        out_image = out_image.toInt16()
                
                # Apply land mask
                if short_region_name in ["est", "bal", "eur"]:
                    out_image = (
                        out_image.where(nodata_mask, nodata_val)
                        .updateMask(land_mask)
                        .unmask(nodata_val)
                    )
                else:
                    out_image = out_image.where(nodata_mask, nodata_val)
                    
                # Loop over grid tile IDs
                for i, grid_id in enumerate(grid_ids):

                    # Output tile ID
                    tile_id = str(i + 1).zfill(2)
                    
                    if tile_id in export_tile_ids:

                        # Get tile geometry
                        feature = ee.Feature(grid_tiles.toList(1, i).get(0))
                        region = feature.geometry()
                        
                        # Launch export task for filled image tile
                        filename = "_".join(
                            [
                                short_region_name,
                                short_ds_name,
                                f"{variable}_{stat}",
                                start_date,
                                end_date,
                                tile_id
                            ]
                        )
                        task_name = f"export_{filename}"
                        long_region_name = get_long_region_name(
                            short_region_name
                        )
                        long_ds_name = get_long_ds_name(
                            short_ds_name
                        )
                        prefix = (
                            f"{long_region_name}/"
                            f"{long_ds_name}/"
                            f"{variable}/"
                            f"{year}/"
                            f"{filename}"
                        )
                        task = ee.batch.Export.image.toCloudStorage(
                            image=out_image,
                            description=task_name,
                            bucket=bucket,
                            fileNamePrefix=prefix,
                            fileFormat="GeoTIFF",
                            crs=region_crs,
                            crsTransform=region_transform,
                            maxPixels=1e13,
                            region=region,
                            formatOptions={
                                "cloudOptimized": True,
                                "noData": nodata_val
                            }
                        )
                        task.start()
                        print(f"\t\t\t\tStarted task: {task_name}")
                        
                        # Launch export task for nodata tile
                        nodata_filename = "_".join(
                            [
                                short_region_name,
                                short_ds_name,
                                f"{variable}",
                                start_date,
                                end_date,
                                tile_id,
                                "nodata"
                            ]
                        )
                        nodata_task_name = f"export_{nodata_filename}"
                        nodata_prefix = (
                            f"{long_region_name}_nodata/"
                            f"{long_ds_name}/"
                            f"{variable}/"
                            f"{year}/"
                            f"{nodata_filename}"
                        )
                        nodata_task = ee.batch.Export.image.toCloudStorage(
                            image=nodata_mask,
                            description=nodata_task_name,
                            bucket=bucket,
                            fileNamePrefix=nodata_prefix,
                            fileFormat="GeoTIFF",
                            crs=region_crs,
                            crsTransform=region_transform,
                            maxPixels=1e13,
                            region=region,
                            formatOptions={
                                "cloudOptimized": True,
                                "noData": 0
                            }
                        )
                        nodata_task.start()
                        print(f"\t\t\t\tStarted task: {nodata_task_name}")
                    
    print("\n")